# FedAvg Experiments Comparison

This notebook compares all available FedAvg experiments (`exp1` to `exp14`) using **all runs found on disk**.

For each experiment or variant, the notebook aggregates the final metrics across runs and reports:
- mean
- sample standard deviation (`std`)
- number of runs (`n_runs`)

The uncertainty shown here is therefore the variability **across repeated runs**, not across clients.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

from reporting import (
    DISPLAY_METRICS,
    EXPERIMENT_ORDER,
    GLOBAL_SORT_ASCENDING,
    GLOBAL_SORT_COLUMNS,
    LOCAL_SORT_ASCENDING,
    LOCAL_SORT_COLUMNS,
    SUMMARY_TXT_PATH,
    aggregate_local_by_client,
    aggregate_local_rounds,
    aggregate_runs,
    export_summary_txt,
    format_mean_std,
    load_run_level_results,
)

cwd = Path.cwd()
if (cwd / 'config.py').exists() and (cwd / 'run_experiment.py').exists():
    fedavg_root = cwd
elif (cwd / 'fedavg-experiments').exists():
    fedavg_root = cwd / 'fedavg-experiments'
else:
    raise FileNotFoundError('Could not locate the fedavg-experiments folder from the current working directory.')

print(f'Using fedavg root: {fedavg_root}')

Using fedavg root: c:\Users\leono\Desktop\iscte\Tese\FL_MasterThesis\fedavg-experiments


## How Splits Are Defined

In [2]:
display(Markdown('''
- **Global train/dev/test:** for most experiments (`exp1`, `exp2`, `exp3`, `exp4`, `exp6`, `exp7`, `exp8`, `exp9`, `exp10`, `exp11`, `exp12`, `exp13`, `exp14`) the setup is **byclient**, so clients are disjoint across train, dev, and test.
- **Cross-dataset case (`exp5`):** one dataset is held out for test, while the other datasets are used for train/dev.
- **Local train/val:** each training client is internally split by trial into a local training subset and a local validation subset.
- **Fine-tuning case (`exp9`):** each test client is additionally split into `adapt` and `eval` subsets for final local fine-tuning.
- **Selection-based cases (`exp6`, `exp10`, `exp13`):** each client compares the current incoming global model with the best previous global model seen by that client, and trains from the better one.
- **Experiment 3 note:** `exp3_local_epochs` is treated as multiple variants (`10`, `25`, `50`, `75`, `100` local epochs) and each variant is aggregated across its own repeated runs.
- **Uncertainty in this notebook:** all uncertainty values correspond to the **sample standard deviation across repeated runs**.
'''))


- **Global train/dev/test:** for most experiments (`exp1`, `exp2`, `exp3`, `exp4`, `exp6`, `exp7`, `exp8`, `exp9`, `exp10`, `exp11`, `exp12`, `exp13`, `exp14`) the setup is **byclient**, so clients are disjoint across train, dev, and test.
- **Cross-dataset case (`exp5`):** one dataset is held out for test, while the other datasets are used for train/dev.
- **Local train/val:** each training client is internally split by trial into a local training subset and a local validation subset.
- **Fine-tuning case (`exp9`):** each test client is additionally split into `adapt` and `eval` subsets for final local fine-tuning.
- **Selection-based cases (`exp6`, `exp10`, `exp13`):** each client compares the current incoming global model with the best previous global model seen by that client, and trains from the better one.
- **Experiment 3 note:** `exp3_local_epochs` is treated as multiple variants (`10`, `25`, `50`, `75`, `100` local epochs) and each variant is aggregated across its own repeated runs.
- **Uncertainty in this notebook:** all uncertainty values correspond to the **sample standard deviation across repeated runs**.


## Load Results

In [3]:
results_df, local_by_client_df, local_round_df = load_run_level_results(fedavg_root)
aggregated_df = aggregate_runs(results_df)
aggregated_local_clients_df = aggregate_local_by_client(local_by_client_df)
aggregated_local_rounds_df = aggregate_local_rounds(local_round_df)
summary_txt_path = export_summary_txt(fedavg_root, fedavg_root / SUMMARY_TXT_PATH.name)

availability = (
    results_df.groupby(['experiment_id', 'comparison_id'], dropna=False)
    .agg(n_runs=('available', lambda s: int(s.sum())), latest_run=('latest_run', 'last'))
    .reset_index()
)
display(availability)

,experiment_id,comparison_id,n_runs,latest_run
0,exp10_clustered_keep_best_local,exp10_clustered_keep_best_local,10,run_20260420_185219
1,exp11_baseline_final,exp11_baseline_final,10,run_20260421_013534
2,exp12_final_unweighted,exp12_final_unweighted,10,run_20260421_081547
3,exp13_final_keep_best_local,exp13_final_keep_best_local,10,run_20260421_152701
4,exp14_final_clustered,exp14_final_clustered,10,run_20260421_222434
5,exp1_fedavg_base,exp1_fedavg_base,10,run_20260415_124639
6,exp2_fraction_clients,exp2_fraction_clients,10,run_20260415_131246
7,exp3_local_epochs,exp3_local_epochs_10,10,run_20260415_151341
8,exp3_local_epochs,exp3_local_epochs_25,10,run_20260415_162950
9,exp3_local_epochs,exp3_local_epochs_50,10,run_20260415_191329


## Final Global Ranking Across Runs

In [4]:
global_rows = []
for _, row in aggregated_df.iterrows():
    global_rows.append({
        'comparison_id': row['comparison_id'],
        'experiment_id': row.get('experiment_id'),
        'title': row.get('title'),
        'n_runs': int(row.get('n_runs', 0)),
        'scenario': row.get('scenario'),
        'num_rounds': row.get('num_rounds'),
        'local_epochs': row.get('local_epochs'),
        'test_accuracy': format_mean_std(row.get('test_accuracy_mean'), row.get('test_accuracy_std')),
        'test_balanced_accuracy': format_mean_std(row.get('test_balanced_accuracy_mean'), row.get('test_balanced_accuracy_std')),
        'test_specificity': format_mean_std(row.get('test_specificity_mean'), row.get('test_specificity_std')),
        'test_precision': format_mean_std(row.get('test_precision_mean'), row.get('test_precision_std')),
        'test_recall': format_mean_std(row.get('test_recall_mean'), row.get('test_recall_std')),
        'test_f1': format_mean_std(row.get('test_f1_mean'), row.get('test_f1_std')),
        'test_roc_auc': format_mean_std(row.get('test_roc_auc_mean'), row.get('test_roc_auc_std')),
        'test_pr_auc': format_mean_std(row.get('test_pr_auc_mean'), row.get('test_pr_auc_std')),
        'test_far': format_mean_std(row.get('test_far_mean'), row.get('test_far_std')),
        'test_miss_rate': format_mean_std(row.get('test_miss_rate_mean'), row.get('test_miss_rate_std')),
    })

global_ranking_df = pd.DataFrame(global_rows)
if not aggregated_df.empty and all(column in aggregated_df.columns for column in GLOBAL_SORT_COLUMNS):
    ordered_ids = aggregated_df.sort_values(GLOBAL_SORT_COLUMNS, ascending=GLOBAL_SORT_ASCENDING)['comparison_id'].tolist()
    global_ranking_df['comparison_id'] = pd.Categorical(global_ranking_df['comparison_id'], categories=ordered_ids, ordered=True)
    global_ranking_df = global_ranking_df.sort_values('comparison_id').reset_index(drop=True)
    global_ranking_df['comparison_id'] = global_ranking_df['comparison_id'].astype(str)

display(global_ranking_df)

,comparison_id,experiment_id,title,n_runs,scenario,num_rounds,local_epochs,test_accuracy,test_balanced_accuracy,test_specificity,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc,test_far,test_miss_rate
0,exp12_final_unweighted,exp12_final_unweighted,Experiment 12 - Final Unweighted,10,byclient,25.0,100.0,0.9493 +/- 0.0026,0.9363 +/- 0.0031,0.9707 +/- 0.0034,0.9330 +/- 0.0072,0.9019 +/- 0.0063,0.9171 +/- 0.0041,0.9839 +/- 0.0010,0.9676 +/- 0.0024,0.0293 +/- 0.0034,0.0981 +/- 0.0063
1,exp11_baseline_final,exp11_baseline_final,Experiment 11 - Baseline Final,10,byclient,25.0,100.0,0.9480 +/- 0.0021,0.9398 +/- 0.0021,0.9614 +/- 0.0040,0.9149 +/- 0.0078,0.9183 +/- 0.0053,0.9166 +/- 0.0032,0.9834 +/- 0.0008,0.9658 +/- 0.0020,0.0386 +/- 0.0040,0.0817 +/- 0.0053
2,exp13_final_keep_best_local,exp13_final_keep_best_local,Experiment 13 - Final Keep-Best Local,10,byclient,25.0,100.0,0.9471 +/- 0.0018,0.9389 +/- 0.0022,0.9607 +/- 0.0032,0.9134 +/- 0.0062,0.9172 +/- 0.0056,0.9153 +/- 0.0027,0.9827 +/- 0.0009,0.9644 +/- 0.0023,0.0393 +/- 0.0032,0.0828 +/- 0.0056
3,exp3_local_epochs_50,exp3_local_epochs,Experiment 3 - Local Epochs Comparison,10,byclient,25.0,50.0,0.9465 +/- 0.0022,0.9382 +/- 0.0020,0.9602 +/- 0.0038,0.9123 +/- 0.0076,0.9163 +/- 0.0047,0.9143 +/- 0.0032,0.9825 +/- 0.0009,0.9642 +/- 0.0017,0.0398 +/- 0.0038,0.0837 +/- 0.0047
4,exp3_local_epochs_75,exp3_local_epochs,Experiment 3 - Local Epochs Comparison,16,byclient,25.0,75.0,0.9468 +/- 0.0024,0.9383 +/- 0.0022,0.9608 +/- 0.0043,0.9135 +/- 0.0085,0.9157 +/- 0.0053,0.9146 +/- 0.0036,0.9826 +/- 0.0008,0.9642 +/- 0.0027,0.0392 +/- 0.0043,0.0843 +/- 0.0053
5,exp14_final_clustered,exp14_final_clustered,Experiment 14 - Final Clustered,10,byclient,25.0,100.0,0.9473 +/- 0.0026,0.9393 +/- 0.0032,0.9605 +/- 0.0036,0.9131 +/- 0.0070,0.9182 +/- 0.0066,0.9156 +/- 0.0041,0.9826 +/- 0.0009,0.9639 +/- 0.0030,0.0395 +/- 0.0036,0.0818 +/- 0.0066
6,exp3_local_epochs_25,exp3_local_epochs,Experiment 3 - Local Epochs Comparison,10,byclient,25.0,25.0,0.9447 +/- 0.0021,0.9361 +/- 0.0022,0.9590 +/- 0.0034,0.9097 +/- 0.0066,0.9132 +/- 0.0050,0.9114 +/- 0.0032,0.9814 +/- 0.0007,0.9610 +/- 0.0022,0.0410 +/- 0.0034,0.0868 +/- 0.0050
7,exp4_unweighted_aggregation,exp4_unweighted_aggregation,Experiment 4 - Unweighted Aggregation,10,byclient,25.0,10.0,0.9393 +/- 0.0020,0.9259 +/- 0.0030,0.9614 +/- 0.0024,0.9125 +/- 0.0049,0.8905 +/- 0.0065,0.9014 +/- 0.0034,0.9789 +/- 0.0007,0.9577 +/- 0.0020,0.0386 +/- 0.0024,0.1095 +/- 0.0065
8,exp9_final_local_finetuning,exp9_final_local_finetuning,Experiment 9 - Final Local Fine-Tuning,10,byclient,25.0,10.0,0.9373 +/- 0.0024,0.9291 +/- 0.0018,0.9509 +/- 0.0048,0.8931 +/- 0.0091,0.9074 +/- 0.0050,0.9002 +/- 0.0033,0.9782 +/- 0.0009,0.9552 +/- 0.0025,0.0491 +/- 0.0048,0.0926 +/- 0.0050
9,exp6_keep_best_local_model,exp6_keep_best_local_model,Experiment 6 - Keep Best Local Model,10,byclient,25.0,10.0,0.9378 +/- 0.0020,0.9298 +/- 0.0019,0.9511 +/- 0.0035,0.8935 +/- 0.0065,0.9085 +/- 0.0041,0.9009 +/- 0.0030,0.9784 +/- 0.0012,0.9548 +/- 0.0034,0.0489 +/- 0.0035,0.0915 +/- 0.0041


## Experiment 3 Local Epochs Comparison Across Runs

In [5]:
exp3_df = aggregated_df[aggregated_df['experiment_id'] == 'exp3_local_epochs'].copy() if not aggregated_df.empty else pd.DataFrame()
exp3_table = pd.DataFrame()
if not exp3_df.empty:
    exp3_table = exp3_df[[
        'comparison_id', 'title', 'n_runs', 'local_epochs', 'num_rounds',
        'test_pr_auc_mean', 'test_pr_auc_std',
        'test_f1_mean', 'test_f1_std',
        'test_balanced_accuracy_mean', 'test_balanced_accuracy_std',
        'test_far_mean', 'test_far_std',
        'test_miss_rate_mean', 'test_miss_rate_std',
    ]].copy()
    exp3_table['test_pr_auc'] = exp3_table.apply(lambda r: format_mean_std(r['test_pr_auc_mean'], r['test_pr_auc_std']), axis=1)
    exp3_table['test_f1'] = exp3_table.apply(lambda r: format_mean_std(r['test_f1_mean'], r['test_f1_std']), axis=1)
    exp3_table['test_balanced_accuracy'] = exp3_table.apply(lambda r: format_mean_std(r['test_balanced_accuracy_mean'], r['test_balanced_accuracy_std']), axis=1)
    exp3_table['test_far'] = exp3_table.apply(lambda r: format_mean_std(r['test_far_mean'], r['test_far_std']), axis=1)
    exp3_table['test_miss_rate'] = exp3_table.apply(lambda r: format_mean_std(r['test_miss_rate_mean'], r['test_miss_rate_std']), axis=1)
    exp3_table = exp3_table[[
        'comparison_id', 'title', 'n_runs', 'local_epochs', 'num_rounds',
        'test_pr_auc', 'test_f1', 'test_balanced_accuracy', 'test_far', 'test_miss_rate'
    ]].sort_values('local_epochs').reset_index(drop=True)

display(exp3_table if not exp3_table.empty else pd.DataFrame(columns=['comparison_id']))

,comparison_id,title,n_runs,local_epochs,num_rounds,test_pr_auc,test_f1,test_balanced_accuracy,test_far,test_miss_rate
0,exp3_local_epochs_10,Experiment 3 - Local Epochs Comparison,10,10.0,25.0,0.9546 +/- 0.0030,0.8996 +/- 0.0046,0.9288 +/- 0.0027,0.0495 +/- 0.0047,0.0929 +/- 0.0045
1,exp3_local_epochs_25,Experiment 3 - Local Epochs Comparison,10,25.0,25.0,0.9610 +/- 0.0022,0.9114 +/- 0.0032,0.9361 +/- 0.0022,0.0410 +/- 0.0034,0.0868 +/- 0.0050
2,exp3_local_epochs_50,Experiment 3 - Local Epochs Comparison,10,50.0,25.0,0.9642 +/- 0.0017,0.9143 +/- 0.0032,0.9382 +/- 0.0020,0.0398 +/- 0.0038,0.0837 +/- 0.0047
3,exp3_local_epochs_75,Experiment 3 - Local Epochs Comparison,16,75.0,25.0,0.9642 +/- 0.0027,0.9146 +/- 0.0036,0.9383 +/- 0.0022,0.0392 +/- 0.0043,0.0843 +/- 0.0053


## Final Local Ranking Across Runs

In [6]:
local_rows = []
for _, row in aggregated_df.iterrows():
    if pd.isna(row.get('local_pr_auc_mean')):
        continue
    local_rows.append({
        'comparison_id': row['comparison_id'],
        'experiment_id': row.get('experiment_id'),
        'title': row.get('title'),
        'n_runs': int(row.get('n_runs', 0)),
        'local_accuracy': format_mean_std(row.get('local_accuracy_mean'), row.get('local_accuracy_std')),
        'local_balanced_accuracy': format_mean_std(row.get('local_balanced_accuracy_mean'), row.get('local_balanced_accuracy_std')),
        'local_specificity': format_mean_std(row.get('local_specificity_mean'), row.get('local_specificity_std')),
        'local_precision': format_mean_std(row.get('local_precision_mean'), row.get('local_precision_std')),
        'local_recall': format_mean_std(row.get('local_recall_mean'), row.get('local_recall_std')),
        'local_f1': format_mean_std(row.get('local_f1_mean'), row.get('local_f1_std')),
        'local_roc_auc': format_mean_std(row.get('local_roc_auc_mean'), row.get('local_roc_auc_std')),
        'local_pr_auc': format_mean_std(row.get('local_pr_auc_mean'), row.get('local_pr_auc_std')),
        'local_far': format_mean_std(row.get('local_far_mean'), row.get('local_far_std')),
        'local_miss_rate': format_mean_std(row.get('local_miss_rate_mean'), row.get('local_miss_rate_std')),
    })

local_ranking_df = pd.DataFrame(local_rows)
if not local_ranking_df.empty and all(column in aggregated_df.columns for column in LOCAL_SORT_COLUMNS):
    ordered_ids = aggregated_df.dropna(subset=['local_pr_auc_mean']).sort_values(LOCAL_SORT_COLUMNS, ascending=LOCAL_SORT_ASCENDING)['comparison_id'].tolist()
    local_ranking_df['comparison_id'] = pd.Categorical(local_ranking_df['comparison_id'], categories=ordered_ids, ordered=True)
    local_ranking_df = local_ranking_df.sort_values('comparison_id').reset_index(drop=True)
    local_ranking_df['comparison_id'] = local_ranking_df['comparison_id'].astype(str)

display(local_ranking_df if not local_ranking_df.empty else pd.DataFrame(columns=['comparison_id']))

,comparison_id


## Aggregated Local Metrics By Client

In [7]:
client_columns = [
    'comparison_id', 'experiment_id', 'dataset', 'client', 'cluster_id', 'evaluation_split', 'model_source',
    'accuracy_mean', 'accuracy_std',
    'balanced_accuracy_mean', 'balanced_accuracy_std',
    'f1_mean', 'f1_std',
    'pr_auc_mean', 'pr_auc_std',
    'far_mean', 'far_std',
    'miss_rate_mean', 'miss_rate_std',
]
client_columns = [c for c in client_columns if c in aggregated_local_clients_df.columns]
display(aggregated_local_clients_df[client_columns] if not aggregated_local_clients_df.empty else pd.DataFrame(columns=['comparison_id']))

,comparison_id


## Aggregated Local Metrics By Round

In [8]:
round_columns = [
    'comparison_id', 'experiment_id', 'round',
    'val_pr_auc_mean', 'val_pr_auc_std',
    'val_f1_mean', 'val_f1_std',
    'val_balanced_accuracy_mean', 'val_balanced_accuracy_std',
    'val_far_mean', 'val_far_std',
    'val_miss_rate_mean', 'val_miss_rate_std',
    'participating_clients_mean', 'participating_clients_std',
    'total_examples_mean', 'total_examples_std',
]
round_columns = [c for c in round_columns if c in aggregated_local_rounds_df.columns]
display(aggregated_local_rounds_df[round_columns] if not aggregated_local_rounds_df.empty else pd.DataFrame(columns=['comparison_id']))

,comparison_id,experiment_id,round,val_pr_auc_mean,val_pr_auc_std,val_f1_mean,val_f1_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_far_mean,val_far_std,val_miss_rate_mean,val_miss_rate_std,participating_clients_mean,participating_clients_std,total_examples_mean,total_examples_std
0,exp11_baseline_final,exp11_baseline_final,1,0.919306,0.007357,0.804728,0.007902,0.929628,0.003953,0.066607,0.005543,0.074137,0.005891,69.0,0.0,41768.4,436.775101
1,exp11_baseline_final,exp11_baseline_final,2,0.929075,0.006759,0.818648,0.008048,0.941272,0.003114,0.060137,0.007141,0.057320,0.003854,69.0,0.0,41768.4,436.775101
2,exp11_baseline_final,exp11_baseline_final,3,0.932654,0.007629,0.822669,0.008640,0.944727,0.003156,0.054794,0.006597,0.055752,0.003552,69.0,0.0,41768.4,436.775101
3,exp11_baseline_final,exp11_baseline_final,4,0.935069,0.008217,0.824838,0.008400,0.946339,0.003268,0.051965,0.006473,0.055357,0.004204,69.0,0.0,41768.4,436.775101
4,exp11_baseline_final,exp11_baseline_final,5,0.936291,0.007581,0.826613,0.008311,0.947562,0.003423,0.049849,0.005896,0.055026,0.004756,69.0,0.0,41768.4,436.775101
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,exp9_final_local_finetuning,exp9_final_local_finetuning,21,0.928249,0.006268,0.819079,0.005927,0.943911,0.002382,0.055460,0.004727,0.056718,0.004498,69.0,0.0,41768.4,436.775101
296,exp9_final_local_finetuning,exp9_final_local_finetuning,22,0.928121,0.006535,0.819643,0.005698,0.944639,0.002302,0.055529,0.004445,0.055193,0.003679,69.0,0.0,41768.4,436.775101
297,exp9_final_local_finetuning,exp9_final_local_finetuning,23,0.928769,0.006792,0.820123,0.005895,0.944819,0.002555,0.054804,0.004150,0.055557,0.004249,69.0,0.0,41768.4,436.775101
298,exp9_final_local_finetuning,exp9_final_local_finetuning,24,0.929488,0.006385,0.820658,0.006097,0.944904,0.002497,0.053992,0.004333,0.056200,0.003907,69.0,0.0,41768.4,436.775101


## Experiment Summary TXT

In [9]:
print(f'Summary TXT written to: {summary_txt_path}')
preview_lines = summary_txt_path.read_text(encoding='utf-8').splitlines()[:80] if summary_txt_path.exists() else ['Summary file not found.']
display(Markdown('```text\n' + '\n'.join(preview_lines) + '\n```'))

Summary TXT written to: c:\Users\leono\Desktop\iscte\Tese\FL_MasterThesis\fedavg-experiments\fedavg_experiments_summary.txt


```text
FedAvg Experiments Summary
==========================

Results are aggregated across all available runs for each experiment/variant.
Uncertainty is reported as sample standard deviation across runs (mean +/- std).

exp12_final_unweighted
----------------------
Title: Experiment 12 - Final Unweighted
Description: Final baseline with 100 local epochs plus unweighted aggregation across participating clients.
Runs aggregated: 10
Scenario: byclient
Global rounds: 25
Local epochs: 100
Fraction fit: 1.0
Weighted aggregation: 0.0
Local model selection: 0.0
Clustered aggregation: 0.0
Personalized head: 0.0
Final local fine-tuning epochs: 0
Random seed in config: 1.0
Model hidden layers: [256, 128, 64]
Model dropout: 0.3
Model learning rate: 0.0008
Model weight decay: 0.0001
Model batch size: 256.0

Final global dev metrics:
  accuracy: 0.9219 +/- 0.0039
  balanced_accuracy: 0.8965 +/- 0.0042
  specificity: 0.9604 +/- 0.0046
  precision: 0.9005 +/- 0.0104
  recall: 0.8327 +/- 0.0070
  f1: 0.8652 +/- 0.0064
  roc_auc: 0.9687 +/- 0.0015
  pr_auc: 0.9357 +/- 0.0054
  far: 0.0396 +/- 0.0046
  miss_rate: 0.1673 +/- 0.0070

Final global test metrics:
  accuracy: 0.9493 +/- 0.0026
  balanced_accuracy: 0.9363 +/- 0.0031
  specificity: 0.9707 +/- 0.0034
  precision: 0.9330 +/- 0.0072
  recall: 0.9019 +/- 0.0063
  f1: 0.9171 +/- 0.0041
  roc_auc: 0.9839 +/- 0.0010
  pr_auc: 0.9676 +/- 0.0024
  far: 0.0293 +/- 0.0034
  miss_rate: 0.0981 +/- 0.0063

Final fine-tuned test-client metrics:
  accuracy: n/a
  balanced_accuracy: n/a
  specificity: n/a
  precision: n/a
  recall: n/a
  f1: n/a
  roc_auc: n/a
  pr_auc: n/a
  far: n/a
  miss_rate: n/a


exp11_baseline_final
--------------------
Title: Experiment 11 - Baseline Final
Description: Final FedAvg baseline using the strongest setting found so far: 100 local epochs with standard weighted aggregation.
Runs aggregated: 10
Scenario: byclient
Global rounds: 25
Local epochs: 100
Fraction fit: 1.0
Weighted aggregation: 1.0
Local model selection: 0.0
Clustered aggregation: 0.0
Personalized head: 0.0
Final local fine-tuning epochs: 0
Random seed in config: 1.0
Model hidden layers: [256, 128, 64]
```

## Best Current Configuration

With the current set of repeated runs, the strongest overall result is coming from **`exp12_final_unweighted`**.

This means that the best-performing configuration so far is:
- **scenario:** `byclient`
- **model/input setup:** the shared FedAvg MLP pipeline used in these experiments
- **global rounds:** `25`
- **local epochs:** `100`
- **aggregation:** **unweighted** aggregation across participating clients
- **client participation:** full participation (`fraction_fit = 1.0`)
- **no local model selection, no clustered aggregation, no personalized head, no final fine-tuning**

Interpreting this result: in the current `byclient` federated setting, giving **equal weight to each client** appears to work better than weighting clients by their number of samples. This suggests that preserving balance across heterogeneous clients is more beneficial than letting larger clients dominate the global update.

A useful nuance is that **`exp11_baseline_final`** remains very competitive and tends to produce a slightly lower mean `miss_rate`, while **`exp12_final_unweighted`** leads the ranking on the main global comparison metrics, especially `PR-AUC`.
